# XFakeNet – interactive demo

Type a statement or headline (and optionally add an image). The demo shows the **verdict**, the **words behind it**
(SHAP), the **most suspicious part**, the **modality flag**, the **C-score**, and the **deception pattern**.

**Steps:** `Runtime → Change runtime type → T4 GPU` (or **Connect without GPU**: it works, about 3–5 s per answer),
then `Runtime → Run all` and allow Google Drive. After ~3 minutes the last cell prints a link like
`https://xxxx.gradio.live`: open it, or share it (it works for 72 hours while this tab stays open).

## 1. Google Drive and trained models

In [ ]:
import os, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none (demo still works, a few seconds per answer)")
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/xfakenet"
os.chdir(WORK)
need = ["runs/xfakenet_v2/xfakenet.pt", "runs/modality.joblib", "runs/deception_clf.joblib",
        "results/cscore/cscore_results.json", "data/processed/corpus_test.csv"]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    raise SystemExit(f"Missing files: {missing}")
print("Models found.")

## 2. Write the project code

In [ ]:
import os
for d in ['configs','src/xfakenet']:
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile configs/week1.yaml
seed: 42
paths:
  raw: data/raw
  processed: data/processed
  images: data/images
  runs: runs

liar:
  dir: data/raw/liar            # expects train.tsv / valid.tsv / test.tsv

extra_classes:
  enabled: false      # three-class setup: LIAR only
  source: tshp17
  file: data/raw/tshp17/fulltrain.csv   # columns: label,text (no header)
  test_file: data/raw/tshp17/balancedtest.csv
  use_hoax_trusted: false
  # Articles are long; LIAR statements are ~18 words. Truncate to the first
  # sentences up to this many words so length is not a shortcut for the class.
  max_words: 40
  # Cap per class so Satire/Propaganda stay minority classes (as in the outline).
  max_per_class: 1500

fakenewsnet:
  dir: data/raw/FakeNewsNet/dataset
  sample_size: 1000            # articles to try for images (balanced fake/real)
  min_image_side: 100          # px; smaller images (icons, pixels) are discarded
  workers: 16
  timeout: 10
  use_wayback: true

baseline:
  model: roberta-base
  max_len: 64
  epochs: 4
  batch_size: 32
  lr: 2.0e-5
  warmup_ratio: 0.06
  weight_decay: 0.01
  fp16: true

# ---------------- Week 2 ----------------
augmentation:                  # final training mix (week 3): 70% original / 15% / 15%
  original_share: 70
  textfooler_share: 15
  paraphrase_share: 15

cscore:
  clip_model: openai/clip-vit-base-patch32
  batch_size: 64
  percentile: 95               # alternative threshold: 95th percentile of aligned C-scores (valid)
  out_dir: results/cscore

paraphrase:
  models: [gemini-3.5-flash-lite, gemini-3.8-flash, gemini-3.1-flash-lite, gemini-2.5-flash]
  batch_size: 20               # statements per API call
  temperature: 0.9
  sleep_s: 4                   # pause between calls (free-tier rate limits)
  test_n: 500
  out_train: data/augment/paraphrase_train.jsonl
  out_test: data/augment/paraphrase_test.jsonl

textfooler:
  victim_dir: runs/baseline/best
  embeddings_url: https://github.com/nmrksic/counter-fitting/raw/master/word_vectors/counter-fitted-vectors.txt.zip
  embeddings_path: data/raw/counter-fitted-vectors.txt.zip
  sim_model: sentence-transformers/all-MiniLM-L6-v2
  k: 50                        # synonyms per word (paper: 50)
  sim_threshold: 0.80
  max_change: 0.25             # at most 25% of words replaced
  query_budget: 2000
  test_n: 500
  out_train: data/augment/textfooler_train.jsonl
  out_test: data/augment/textfooler_test.jsonl

model_v1:
  lrs: [1.0e-5, 2.0e-5, 3.0e-5]
  epochs: 5

# ---------------- Week 3 ----------------
xfakenet:                      # RoBERTa + jointly trained token/segment attention
  encoder: roberta-base
  max_len: 64
  max_segments: 8
  att_dim: 256
  epochs: 4
  batch_size: 32
  lr: 2.0e-5                   # encoder
  head_lr: 1.0e-4              # attention layers + classifier
  warmup_ratio: 0.06
  weight_decay: 0.01
  highlight_threshold: 0.65    # outline: tokens above T = 0.65 are highlighted

modality:
  C: 0.1                       # regularisation of the text/image logistic models (389 items)

week3:
  out_dir: results/week3

# ---------------- Week 4 ----------------
week4:
  out_dir: results/week4

deception:
  n_mismatched: 200            # shuffled pairs added to the 389 real ones for labelling
  batch_size: 5                # image-text pairs per Gemini call
  image_side: 512
  sleep_s: 4
  C: 0.5
  min_per_class: 5
  handcheck_n: 100
  labels: data/deception/gemini_labels.jsonl
  features: data/deception/clip_features.npz
  oof: data/deception/classifier_oof.csv
  model: runs/deception_clf.joblib
  handcheck_sample: data/deception/handcheck_sample.csv
  handcheck_results: data/deception/handcheck_results.csv

pipeline:
  modality_model: runs/modality.joblib

explain:                       # faithful explanations (week 4 fix)
  permutations_gpu: 10
  permutations_cpu: 6
  cpu_statements: 400
  chunk: 50

In [ ]:
%%writefile configs/label_mapping.yaml
# XFakeNet five-class label mapping (frozen 2026-10-21, Week 1)
# Every later step (training, augmentation, evaluation) reads this file.
# Change it only with a note in docs/label_mapping.md and a full retrain.

# Three-class setup (changed 20 Oct 2026): Satire/Propaganda dropped because the
# only available source (TSHP-17) taught the model website style, not the class
# (Propaganda F1 0.99 on same-site validation, 0.08 on unseen-site test).
# For the old five-class setup use:
#   classes: [Fabricated, Misleading, Satire, Propaganda, Credible]
#   and set extra_classes.enabled: true in week1.yaml
classes: [Fabricated, Misleading, Credible]   # index order = model label ids

liar:
  # LIAR's six PolitiFact truth ratings -> XFakeNet classes
  # Keys are quoted on purpose: unquoted true/false become YAML booleans.
  "pants-fire":  Fabricated
  "false":       Fabricated
  "barely-true": Misleading
  "half-true":   Misleading
  "mostly-true": Credible
  "true":        Credible

# LIAR has no Satire or Propaganda signal. These two classes come from an
# external labelled source (default: TSHP-17, Rashkin et al. 2017).
tshp17:
  # TSHP-17 integer labels: 1=Satire 2=Hoax 3=Propaganda 4=Trusted
  1: Satire
  3: Propaganda
  # 2 (Hoax) and 4 (Trusted) are NOT used by default: they would add
  # long-article Fabricated/Credible examples whose style differs from LIAR.
  # Set use_hoax_trusted: true in configs/week1.yaml to include them.
  2: Fabricated
  4: Credible

fakenewsnet:
  # Binary only; used for C-score calibration/evaluation, not five-class training.
  "fake": Fabricated
  "real": Credible

In [ ]:
%%writefile src/xfakenet/__init__.py
# package marker

In [ ]:
%%writefile src/xfakenet/common.py
"""Shared helpers: config loading, label mapping, seeding, text cleanup."""
from __future__ import annotations

import json
import random
import re
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]


def load_yaml(path: str | Path) -> dict:
    with open(ROOT / path if not Path(path).is_absolute() else path) as f:
        return yaml.safe_load(f)


def load_config() -> dict:
    return load_yaml("configs/week1.yaml")


def load_mapping() -> dict:
    return load_yaml("configs/label_mapping.yaml")


def class_names() -> list[str]:
    return load_mapping()["classes"]


def label2id() -> dict[str, int]:
    return {c: i for i, c in enumerate(class_names())}


def p(rel: str | Path) -> Path:
    """Resolve a repo-relative path."""
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_WS = re.compile(r"\s+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return _WS.sub(" ", s).strip()


def truncate_words(text: str, max_words: int) -> str:
    """Keep whole leading sentences up to max_words; hard-cut if the first sentence is longer."""
    text = clean_text(text)
    out, n = [], 0
    for sent in _SENT.split(text):
        w = sent.split()
        if n + len(w) > max_words:
            if not out:
                out.append(" ".join(w[:max_words]))
            break
        out.append(sent)
        n += len(w)
    return " ".join(out)


def write_json(obj, path: str | Path) -> None:
    with open(p(path), "w") as f:
        json.dump(obj, f, indent=2, default=str)

In [ ]:
%%writefile src/xfakenet/segments.py
"""Split a statement into segments (sentences and clauses) for the sentence-attention layer.

About 81% of LIAR statements are a single sentence, so splitting only at sentence
boundaries would give the attention layer nothing to choose between. We also split
at ; : commas and dashes, then merge pieces shorter than `min_words` into their
neighbour so a segment is always a meaningful clause.
"""
from __future__ import annotations

import re

_SPLIT = re.compile(r"(?<=[.!?;:])\s+|,\s+|\s+--?\s+")


def segment_spans(text: str, max_segments: int = 8, min_words: int = 3) -> list[tuple[int, int]]:
    """Return character spans (start, end) of the segments of `text`."""
    spans, start = [], 0
    for m in _SPLIT.finditer(text):
        if m.start() > start:
            spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    if not spans:
        return [(0, len(text))]

    words = lambda s: len(text[s[0]:s[1]].split())  # noqa: E731
    merged: list[tuple[int, int]] = []
    for s in spans:
        if merged and (words(s) < min_words or words(merged[-1]) < min_words):
            merged[-1] = (merged[-1][0], s[1])
        else:
            merged.append(s)
    if len(merged) > max_segments:
        merged = merged[: max_segments - 1] + [(merged[max_segments - 1][0], merged[-1][1])]
    return merged


def segments(text: str, max_segments: int = 8, min_words: int = 3) -> list[str]:
    return [text[a:b] for a, b in segment_spans(text, max_segments, min_words)]

In [ ]:
%%writefile src/xfakenet/explain_utils.py
"""Word- and segment-level explanation from attention weights (no torch needed)."""
from __future__ import annotations

from bisect import bisect_right

import numpy as np


def word_explanation(text, offsets, seg_ids, tok_att, seg_att, spans, probs, classes, thr) -> dict:
    """Turn token/segment attention into word-level and segment-level explanations (pure numpy)."""
    words, w_imp, pos = [], [], 0
    for w in text.split():
        a = text.index(w, pos)
        words.append((a, a + len(w), w))
        w_imp.append(0.0)
        pos = a + len(w)
    starts = [a for a, _, _ in words]
    for j, (a, z) in enumerate(offsets):
        if z > a and seg_ids[j] >= 0 and words:
            k = max(0, bisect_right(starts, a) - 1)
            w_imp[k] += float(tok_att[j])      # a word's importance = sum over its sub-word tokens
    m = max(w_imp) if w_imp and max(w_imp) > 0 else 1.0
    norm = [x / m for x in w_imp]
    segs = [{"text": text[a:z], "weight": round(float(seg_att[k]), 4)}
            for k, (a, z) in enumerate(spans) if k < len(seg_att)]
    top = int(np.argmax([s["weight"] for s in segs])) if segs else 0
    return {
        "text": text,
        "pred": classes[int(np.argmax(probs))],
        "probs": {c: round(float(p_), 4) for c, p_ in zip(classes, probs)},
        "words": [{"w": w, "weight": round(x, 4)} for (_, _, w), x in zip(words, norm)],
        "highlighted": [w for (_, _, w), x in zip(words, norm) if x >= thr],
        "segments": segs,
        "top_segment": segs[top]["text"] if segs else text,
    }

In [ ]:
%%writefile src/xfakenet/model.py
"""XFakeNet classifier: RoBERTa + jointly trained token- and segment-attention (Module 1 + 3).

    tokens --RoBERTa--> h_t
    token attention (within each segment):   a_t = softmax_t( v_t . tanh(W_t h_t) )
    segment vector:                          s_k = sum_t a_t h_t
    segment attention:                       alpha_k = softmax_k( v_s . tanh(W_s s_k) )
    statement vector:                        d = sum_k alpha_k s_k
    prediction:                              softmax(W_c d)

The classifier only sees d, so the attention weights are part of the decision
itself (explanation produced in the forward pass), not computed afterwards.
Token importance shown to the user = a_t * alpha_(segment of t).
"""
from __future__ import annotations

import json
from bisect import bisect_right
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
from torch import nn

from .explain_utils import word_explanation
from .segments import segment_spans


class XFakeNetModel(nn.Module):
    def __init__(self, encoder: nn.Module, num_labels: int, max_segments: int = 8,
                 att_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.encoder = encoder
        d = encoder.config.hidden_size
        self.K = max_segments
        self.tok_proj, self.tok_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.seg_proj, self.seg_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.cls = nn.Linear(d, num_labels)

    def forward(self, input_ids, attention_mask, seg_ids):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # B L D
        b, l, _ = h.shape
        e = self.tok_v(torch.tanh(self.tok_proj(h))).squeeze(-1).float()                       # B L
        ks = torch.arange(self.K, device=h.device)
        member = seg_ids.unsqueeze(1) == ks.view(1, -1, 1)                                       # B K L
        scores = e.unsqueeze(1).expand(b, self.K, l).masked_fill(~member, -1e4)
        a = torch.softmax(scores, -1) * member                                                   # B K L (empty segment -> 0)
        s = torch.bmm(a.to(h.dtype), h)                                                          # B K D
        valid = member.any(-1)                                                                   # B K
        u = self.seg_v(torch.tanh(self.seg_proj(s))).squeeze(-1).float().masked_fill(~valid, -1e4)
        alpha = torch.softmax(u, -1) * valid                                                     # B K
        doc = torch.bmm(alpha.unsqueeze(1).to(s.dtype), s).squeeze(1)                            # B D
        logits = self.cls(self.drop(doc)).float()
        token_imp = (a * alpha.unsqueeze(-1)).sum(1)                                             # B L
        return {"logits": logits, "token_att": token_imp, "seg_att": alpha}


def encode(tok, texts: Sequence[str], max_len: int, max_segments: int, device=None):
    """Tokenise and assign every token to a segment (special/pad tokens get -1)."""
    enc = tok(list(texts), truncation=True, max_length=max_len, padding=True,
              return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")
    seg_ids = torch.full(enc["input_ids"].shape, -1, dtype=torch.long)
    spans_all = []
    for i, text in enumerate(texts):
        spans = segment_spans(text, max_segments)
        starts = [a for a, _ in spans]
        spans_all.append(spans)
        for j, (a, z) in enumerate(offsets[i].tolist()):
            if z > a:
                seg_ids[i, j] = max(0, bisect_right(starts, a) - 1)
    enc["seg_ids"] = seg_ids
    if device is not None:
        enc = {k: v.to(device) for k, v in enc.items()}
    return enc, offsets, spans_all


def build(encoder_name: str, num_labels: int, max_segments: int, att_dim: int) -> XFakeNetModel:
    from transformers import AutoModel
    return XFakeNetModel(AutoModel.from_pretrained(encoder_name), num_labels, max_segments, att_dim)


def save(model: XFakeNetModel, tok, out_dir: str | Path, meta: dict) -> None:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out / "xfakenet.pt")
    model.encoder.config.save_pretrained(out / "encoder_config")
    tok.save_pretrained(out)
    json.dump(meta, open(out / "xfakenet_meta.json", "w"), indent=2)


def load(model_dir: str | Path, device: str = "cpu"):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    d = Path(model_dir)
    meta = json.load(open(d / "xfakenet_meta.json"))
    enc = AutoModel.from_config(AutoConfig.from_pretrained(d / "encoder_config"))
    model = XFakeNetModel(enc, len(meta["classes"]), meta["max_segments"], meta["att_dim"])
    model.load_state_dict(torch.load(d / "xfakenet.pt", map_location="cpu"))
    return model.to(device).eval(), AutoTokenizer.from_pretrained(d), meta


class Predictor:
    """Batch prediction + explanations for a saved XFakeNet model."""

    def __init__(self, model_dir: str | Path, batch_size: int = 64, device: str | None = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.tok, self.meta = load(model_dir, self.device)
        self.bs = batch_size

    @torch.no_grad()
    def _run(self, texts: Sequence[str]):
        for s in range(0, len(texts), self.bs):
            batch = list(texts[s:s + self.bs])
            enc, offsets, spans = encode(self.tok, batch, self.meta["max_len"], self.meta["max_segments"],
                                         self.device)
            with torch.autocast(device_type="cuda", enabled=self.device == "cuda"):
                out = self.model(**enc)
            yield batch, enc, offsets, spans, out

    def predict_proba(self, texts: Sequence[str]) -> np.ndarray:
        res = [torch.softmax(o["logits"], -1).cpu().numpy() for *_, o in self._run(texts)]
        return np.concatenate(res) if res else np.zeros((0, len(self.meta["classes"])))

    def explain(self, texts: Sequence[str], threshold: float | None = None) -> list[dict]:
        thr = self.meta.get("highlight_threshold", 0.65) if threshold is None else threshold
        out = []
        for batch, enc, offsets, spans, o in self._run(texts):
            probs = torch.softmax(o["logits"], -1).cpu().numpy()
            tok_att = o["token_att"].float().cpu().numpy()
            seg_att = o["seg_att"].float().cpu().numpy()
            seg_ids = enc["seg_ids"].cpu().numpy()
            offs = offsets.tolist()
            for i, text in enumerate(batch):
                out.append(word_explanation(text, offs[i], seg_ids[i], tok_att[i], seg_att[i], spans[i],
                                            probs[i], self.meta["classes"], thr))
        return out

In [ ]:
%%writefile src/xfakenet/attribution.py
"""Faithful (perturbation-based) explanations for any text classifier.

Week 4 showed that the built-in attention weights are not faithful: removing the
highlighted words hurts the prediction barely more than removing random words, and
the attention top segment agrees with SHAP only at chance level. So the explanations
shown to the user are now computed from the model's actual behaviour:

- word_shap     : Shapley values of each word, estimated by permutation sampling
                  (Strumbelj & Kononenko 2014; the estimator behind SHAP's sampling explainer).
                  A word's value = its average change in the predicted-class probability when
                  added to random subsets of the other words.
- occlusion     : leave-one-out: drop in predicted-class probability when the word is deleted.
- segment_shap  : exact Shapley values over segments (all 2^K subsets, K <= 8).

All functions batch every text they need into ONE call to `predict_proba`, so they are
fast on a GPU. Removed words/segments are deleted from the text.
"""
from __future__ import annotations

import itertools
import math
from typing import Callable, Sequence

import numpy as np

Predict = Callable[[Sequence[str]], np.ndarray]


def _join(words: list[str], keep) -> str:
    return " ".join(w for i, w in enumerate(words) if i in keep)


def plan_word_shap(words: list[str], m: int, rng: np.random.Generator):
    """Texts needed for permutation Shapley + bookkeeping to turn their scores into values."""
    n = len(words)
    texts, steps = [""], []          # texts[0] = empty statement (shared start of every chain)
    for _ in range(m):
        perm = rng.permutation(n)
        prev, keep = 0, set()
        for j in perm:
            keep.add(int(j))
            texts.append(_join(words, keep))
            steps.append((int(j), prev, len(texts) - 1))   # value(j) += p[cur] - p[prev]
            prev = len(texts) - 1
    return texts, steps


def finish_word_shap(n: int, m: int, steps, p: np.ndarray) -> np.ndarray:
    phi = np.zeros(n)
    for j, a, b in steps:
        phi[j] += p[b] - p[a]
    return phi / max(1, m)


def plan_occlusion(words: list[str]):
    return [_join(words, set(range(len(words))) - {i}) for i in range(len(words))]


def plan_segment_shap(segs: list[str]):
    k = len(segs)
    subsets = [S for r in range(k + 1) for S in itertools.combinations(range(k), r)]
    return [" ".join(segs[j] for j in S) for S in subsets], subsets


def finish_segment_shap(k: int, subsets, p: np.ndarray) -> np.ndarray:
    v = dict(zip(subsets, p))
    phi = np.zeros(k)
    for i in range(k):
        others = [j for j in range(k) if j != i]
        for r in range(len(others) + 1):
            w = math.factorial(r) * math.factorial(k - r - 1) / math.factorial(k)
            for S in itertools.combinations(others, r):
                phi[i] += w * (v[tuple(sorted(S + (i,)))] - v[S])
    return phi


def explain_batch(predict: Predict, items: list[dict], classes: list[str], m: int = 8,
                  seed: int = 42, methods=("shap", "occlusion", "segment")) -> list[dict]:
    """items: [{"words": [...], "segments": [...], "pred": class name}] -> adds scores per method.

    One predict_proba call for everything (chunk `items` yourself to bound memory)."""
    rng = np.random.default_rng(seed)
    all_texts, plans = [], []
    for it in items:
        words, plan = it["words"], {}
        if "shap" in methods and words:
            t, steps = plan_word_shap(words, m, rng)
            plan["shap"] = (len(all_texts), len(t), steps)
            all_texts += t
        if "occlusion" in methods and words:
            t = plan_occlusion(words)
            plan["occ"] = (len(all_texts), len(t))
            all_texts += t + [" ".join(words)]
        if "segment" in methods and 2 <= len(it["segments"]) <= 8:
            t, subsets = plan_segment_shap(it["segments"])
            plan["seg"] = (len(all_texts), len(t), subsets)
            all_texts += t
        plans.append(plan)
    probs = predict(all_texts) if all_texts else np.zeros((0, len(classes)))
    out = []
    for it, plan in zip(items, plans):
        c = classes.index(it["pred"])
        r = {}
        if "shap" in plan:
            s, n, steps = plan["shap"]
            r["shap"] = finish_word_shap(len(it["words"]), m, steps, probs[s:s + n, c]).tolist()
        if "occ" in plan:
            s, n = plan["occ"]
            full = probs[s + n, c]
            r["occlusion"] = (full - probs[s:s + n, c]).tolist()
        if "seg" in plan:
            s, n, subsets = plan["seg"]
            r["segment_shap"] = finish_segment_shap(len(it["segments"]), subsets, probs[s:s + n, c]).tolist()
        out.append(r)
    return out


def highlight(words: list[str], scores: Sequence[float], thr: float = 0.65) -> list[str]:
    """Words whose positive contribution is >= thr x the largest positive contribution."""
    s = np.asarray(scores, dtype=float)
    top = s.max() if len(s) else 0.0
    if top <= 0:
        return []
    return [w for w, x in zip(words, s) if x > 0 and x / top >= thr]

In [ ]:
%%writefile src/xfakenet/clip_utils.py
"""CLIP ViT-B/32 helpers shared by the C-score, modality flag, deception classifier and pipeline."""
from __future__ import annotations

import numpy as np


class Clip:
    def __init__(self, name: str = "openai/clip-vit-base-patch32", device: str | None = None):
        import torch
        from transformers import CLIPModel, CLIPProcessor
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(name).to(self.device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)

    def embed(self, titles: list[str], images: list, batch_size: int = 64):
        """Normalised (text, image) embeddings. `images` are PIL images or file paths."""
        from PIL import Image
        T, I = [], []
        for s in range(0, len(titles), batch_size):
            ims = [Image.open(x).convert("RGB") if isinstance(x, (str, bytes)) or hasattr(x, "__fspath__")
                   else x.convert("RGB") for x in images[s:s + batch_size]]
            inp = self.proc(text=list(titles[s:s + batch_size]), images=ims, return_tensors="pt",
                            padding=True, truncation=True, max_length=77).to(self.device)
            with self.torch.no_grad():
                out = self.model(**inp)
            T.append(self.torch.nn.functional.normalize(out.text_embeds, dim=-1).cpu().numpy())
            I.append(self.torch.nn.functional.normalize(out.image_embeds, dim=-1).cpu().numpy())
        return np.concatenate(T), np.concatenate(I)


def pair_features(T: np.ndarray, I: np.ndarray) -> np.ndarray:
    """Features for the deception-pattern classifier: text, image, their product, and the C-score."""
    c = 1.0 - (T * I).sum(1, keepdims=True)
    return np.hstack([T, I, T * I, c])

In [ ]:
%%writefile src/xfakenet/pipeline.py
"""The integrated XFakeNet pipeline: one call returns class, explanations, modality flag and pattern.

    pipe = XFakeNetPipeline.load()
    out = pipe("Headline or statement text", image="path/or/PIL.Image")   # image optional

Output keys:
  label, probs                      classifier decision (Fabricated / Misleading / Credible)
  highlighted, word_scores,         Module 1 explanations. Highlights and the top segment come from
  segments, top_segment             SHAP (faithful); the built-in attention versions are kept as
  attention_highlighted,            attention_highlighted / attention_top_segment for comparison.
  attention_top_segment
  modality_flag                     TEXT / IMAGE / CROSS-MODAL (TEXT when no image is given)
  cscore, cscore_threshold          Module 2 contradiction score (only with an image)
  deception_pattern, pattern_probs  one of six patterns (only when CROSS-MODAL)
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from .common import load_config, p


class XFakeNetPipeline:
    def __init__(self, model_dir, cscore_threshold, modality_path=None, deception_path=None,
                 clip_name="openai/clip-vit-base-patch32", shap_permutations: int = 10, threshold: float = 0.65):
        from .model import Predictor
        self.pred = Predictor(model_dir)
        self.thr = float(cscore_threshold)
        self.m, self.hl_thr = shap_permutations, threshold
        self.clip_name = clip_name
        self._clip = None
        self.modality = self._load(modality_path)
        self.deception = self._load(deception_path)

    @staticmethod
    def _load(path):
        if path and Path(path).exists():
            import joblib
            return joblib.load(path)
        return None

    @classmethod
    def load(cls, model_dir: str = "runs/xfakenet_v2"):
        cfg = load_config()
        thr = json.load(open(p(f"{cfg['cscore']['out_dir']}/cscore_results.json")))["chosen_threshold"]
        return cls(p(model_dir), thr, p(cfg["pipeline"]["modality_model"]), p(cfg["deception"]["model"]),
                   cfg["cscore"]["clip_model"], cfg["explain"]["permutations_gpu"],
                   cfg["xfakenet"]["highlight_threshold"])

    @property
    def clip(self):
        if self._clip is None:
            from .clip_utils import Clip
            self._clip = Clip(self.clip_name)
        return self._clip

    def __call__(self, text: str, image=None) -> dict:
        from .attribution import explain_batch, highlight
        e = self.pred.explain([text])[0]
        words, segs = [w["w"] for w in e["words"]], [s["text"] for s in e["segments"]]
        a = explain_batch(self.pred.predict_proba, [{"words": words, "segments": segs, "pred": e["pred"]}],
                          self.pred.meta["classes"], m=self.m, methods=("shap", "segment"))[0]
        shap = a.get("shap", [0.0] * len(words))
        seg_sc = a.get("segment_shap") or [s["weight"] for s in e["segments"]]
        top_seg = segs[int(np.argmax(seg_sc))] if segs else text
        out = {"label": e["pred"], "probs": e["probs"],
               "highlighted": highlight(words, shap, self.hl_thr),
               "word_scores": [{"w": w, "shap": round(float(x), 4)} for w, x in zip(words, shap)],
               "segments": [{"text": s, "shap": round(float(x), 4)} for s, x in zip(segs, seg_sc)],
               "top_segment": top_seg,
               "attention_highlighted": e["highlighted"], "attention_top_segment": e["top_segment"],
               "modality_flag": "TEXT", "cscore": None, "cscore_threshold": self.thr,
               "deception_pattern": None, "pattern_probs": None}
        if image is None:
            return out
        from .clip_utils import pair_features
        T, I = self.clip.embed([text], [image])
        c = float(1.0 - (T * I).sum())
        out["cscore"] = round(c, 4)
        if c > self.thr:
            out["modality_flag"] = "CROSS-MODAL"
            if self.deception is not None:
                pr = self.deception["model"].predict_proba(pair_features(T, I))[0]
                cls = list(self.deception["model"].classes_)
                out["pattern_probs"] = {k: round(float(v), 4) for k, v in zip(cls, pr)}
                out["deception_pattern"] = cls[int(np.argmax(pr))]
        elif self.modality is not None:
            m = self.modality
            zt = float(m["text"].decision_function(T)[0])
            zi = float(m["image"].decision_function(I)[0])
            z = m["fusion_scaler"].transform([[zt, zi]])[0]
            ct, ci = m["fusion_coef"][0] * z[0], m["fusion_coef"][1] * z[1]
            out["modality_flag"] = "TEXT" if abs(ct) >= abs(ci) else "IMAGE"
            out["modality_contrib"] = {"text": round(float(ct), 4), "image": round(float(ci), 4)}
        return out

In [ ]:
%%writefile src/xfakenet/demo.py
"""Gradio demo for XFakeNet.

    from xfakenet.demo import build_app
    build_app(XFakeNetPipeline.load(), examples).launch(share=True)

Every colour is set inline on every element, so Gradio's theme (light or dark) cannot
override it, and the app forces Gradio's light theme. The render functions are plain
Python (no Gradio) so they can be tested anywhere.
"""
from __future__ import annotations

import html

INK, MUTED, FAINT, LINE, CARD = "#1c1917", "#57534e", "#a8a29e", "#e7e5e4", "#ffffff"
FONT = "Inter, -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif"
CLASS_INFO = {
    "Fabricated": ("#b91c1c", "#fee2e2", "False claim with no factual basis"),
    "Misleading": ("#c2410c", "#ffedd5", "Contains some truth but distorts or omits context"),
    "Credible": ("#15803d", "#dcfce7", "Accurate or substantially accurate"),
}
FLAG_INFO = {
    "TEXT": ("#44403c", "#f5f5f4", "The evidence comes mainly from the text."),
    "IMAGE": ("#075985", "#e0f2fe", "The image carries more of the evidence than the text."),
    "CROSS-MODAL": ("#92400e", "#fef3c7", "The image and the headline contradict each other."),
}
PATTERN_INFO = {
    "out_of_context": "Genuine photo from a different event, time or story",
    "misattributed_entity": "Image shows a different person or organisation than named",
    "wrong_location_or_event": "Image shows a different place or kind of event",
    "exaggeration": "Image and headline differ sharply in drama or emotion",
    "manipulated_or_synthetic": "Image looks edited, composited, a meme or AI-generated",
    "irrelevant_stock": "Generic stock image with no real link to the claim",
}


def _e(s) -> str:
    return html.escape(str(s))


def _card(inner: str) -> str:
    return (f'<div style="font-family:{FONT};color:{INK};background:{CARD};border:1px solid {LINE};'
            f'border-radius:14px;padding:20px 22px;box-shadow:0 1px 2px rgba(0,0,0,.04)">{inner}</div>')


def _label(text: str, top: int = 0) -> str:
    return (f'<div style="font-size:12px;font-weight:600;letter-spacing:.06em;text-transform:uppercase;'
            f'color:{MUTED};margin:{top}px 0 8px">{_e(text)}</div>')


def render_verdict(out: dict) -> str:
    color, tint, desc = CLASS_INFO.get(out["label"], (INK, "#f5f5f4", ""))
    rows = ""
    for k, v in sorted(out["probs"].items(), key=lambda kv: -kv[1]):
        c = CLASS_INFO.get(k, (FAINT, "", ""))[0]
        bold = "700" if k == out["label"] else "500"
        rows += (f'<div style="display:flex;align-items:center;gap:12px;margin:8px 0">'
                 f'<span style="width:92px;font-size:14px;font-weight:{bold};color:{INK}">{_e(k)}</span>'
                 f'<span style="flex:1;height:10px;background:#f0eeeb;border-radius:5px;overflow:hidden;display:block">'
                 f'<span style="display:block;height:100%;width:{v*100:.1f}%;background:{c};border-radius:5px"></span></span>'
                 f'<span style="width:44px;text-align:right;font-size:14px;font-weight:{bold};color:{INK}">{v*100:.0f}%</span></div>')
    head = (f'<div style="display:flex;align-items:center;gap:12px;margin-bottom:4px">'
            f'<span style="font-size:28px;font-weight:700;color:{color};line-height:1.2">{_e(out["label"])}</span>'
            f'<span style="font-size:12px;font-weight:600;color:{color};background:{tint};padding:3px 10px;'
            f'border-radius:999px">{out["probs"][out["label"]]*100:.0f}% confident</span></div>')
    return _card(_label("Verdict") + head +
                 f'<div style="font-size:14px;color:{MUTED};margin-bottom:14px">{_e(desc)}</div>' + rows)


def render_modality(out: dict) -> str:
    flag = out["modality_flag"]
    fc, ft, fdesc = FLAG_INFO.get(flag, (INK, "#f5f5f4", ""))
    pill = (f'<span style="display:inline-block;font-size:13px;font-weight:700;color:{fc};background:{ft};'
            f'padding:4px 12px;border-radius:999px;letter-spacing:.03em">{_e(flag)}</span>')
    if out.get("cscore") is None:
        body = (f'<div style="font-size:14px;color:{MUTED};margin-top:10px">No image given, so the analysis is text '
                f'only. Add an image to check whether it fits the headline.</div>')
    else:
        c, t = out["cscore"], out["cscore_threshold"]
        stat = lambda name, val, col=INK: (  # noqa: E731
            f'<div><div style="font-size:12px;color:{MUTED}">{_e(name)}</div>'
            f'<div style="font-size:22px;font-weight:700;color:{col}">{_e(val)}</div></div>')
        verdict = "Contradict" if c > t else "Consistent"
        body = (f'<div style="font-size:14px;color:{MUTED};margin-top:10px">{_e(fdesc)}</div>'
                f'<div style="display:flex;gap:28px;flex-wrap:wrap;margin-top:14px">'
                f'{stat("C-score", f"{c:.3f}")}{stat("Threshold", f"{t:.3f}")}'
                f'{stat("Image vs headline", verdict, "#92400e" if c > t else "#15803d")}</div>')
        pat = out.get("deception_pattern")
        if pat:
            body += (_label("Likely deception pattern", 18) +
                     f'<div style="font-size:15px;color:{INK}"><span style="font-weight:700;color:{INK}">'
                     f'{_e(pat.replace("_", " ").capitalize())}</span>'
                     f'<span style="color:{MUTED}"> · {_e(PATTERN_INFO.get(pat, ""))}</span></div>')
    return _card(_label("Modality") + pill + body)


def render_explanation(out: dict) -> str:
    ws = out.get("word_scores", [])
    top = max([w["shap"] for w in ws if w["shap"] > 0], default=0) or 1.0
    hl = set(out.get("highlighted", []))
    chips = []
    for w in ws:
        a = max(0.0, w["shap"] / top)
        is_hl = w["w"] in hl
        style = (f'display:inline-block;margin:3px 2px;padding:2px 7px;border-radius:6px;font-size:16px;'
                 f'line-height:1.6;color:{INK};background:rgba(234,88,12,{a*0.45:.2f});'
                 f'border:1.5px solid {"#c2410c" if is_hl else "transparent"};font-weight:{"700" if is_hl else "400"}')
        chips.append(f'<span title="SHAP {w["shap"]:+.3f}" style="{style}">{_e(w["w"])}</span>')
    att = ", ".join(_e(w) for w in out.get("attention_highlighted", [])) or "none"
    seg = (f'<div style="font-size:16px;color:{INK};background:#fff7ed;border-left:4px solid #ea580c;'
           f'padding:10px 14px;border-radius:8px">{_e(out["top_segment"])}</div>')
    note = (f'<div style="font-size:12.5px;color:{MUTED};margin-top:14px;line-height:1.5">Darker = pushed harder toward '
            f'the verdict (SHAP). Outlined words are highlighted (at least 65% of the strongest). For comparison, the '
            f'model\'s built-in attention highlighted: {att}.</div>')
    return _card(_label("Why: words that pushed the model to this verdict") + f'<div>{"".join(chips)}</div>' +
                 _label("Most suspicious part", 18) + seg + note)


def render_placeholder(msg: str) -> str:
    return _card(f'<div style="font-size:14px;color:{MUTED}">{_e(msg)}</div>')


FORCE_LIGHT = """() => {
  const u = new URL(window.location);
  if (u.searchParams.get('__theme') !== 'light') { u.searchParams.set('__theme', 'light'); window.location.replace(u.href); }
}"""


def build_app(pipe, examples: list | None = None):
    import gradio as gr

    def run(text, image):
        text = (text or "").strip()
        if not text:
            return render_placeholder("Type a statement or headline first."), "", ""
        out = pipe(text, image=image)
        return render_verdict(out), render_explanation(out), render_modality(out)

    theme = gr.themes.Soft(primary_hue="orange", neutral_hue="stone",
                           font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"])
    look = {"theme": theme, "js": FORCE_LIGHT}
    new_api = int(gr.__version__.split(".")[0]) >= 6      # Gradio 6 moved theme/js from Blocks() to launch()
    with gr.Blocks(title="XFakeNet", **({} if new_api else look)) as app:
        gr.HTML(f'<div style="font-family:{FONT};padding:8px 2px 4px">'
                f'<div style="font-size:30px;font-weight:800;color:{INK}">XFakeNet</div>'
                f'<div style="font-size:15px;color:{MUTED};margin-top:4px">Explainable fake-news detection: a verdict, '
                f'the words behind it, and whether the image fits the headline. Research prototype trained on LIAR '
                f'(US political statements), not a fact-checker.</div></div>')
        with gr.Row(equal_height=False):
            with gr.Column(scale=5):
                text = gr.Textbox(label="Statement or headline", lines=3,
                                  placeholder="e.g. Says the unemployment rate doubled under the governor.")
                image = gr.Image(label="Image (optional)", type="pil", height=260)
                btn = gr.Button("Analyse", variant="primary", size="lg")
            with gr.Column(scale=6):
                verdict = gr.HTML(render_placeholder("The verdict will appear here."))
                modality = gr.HTML()
        explanation = gr.HTML()
        btn.click(run, [text, image], [verdict, explanation, modality])
        text.submit(run, [text, image], [verdict, explanation, modality])
        if examples:
            gr.Examples(examples=examples, inputs=[text, image], label="Examples (click one, then Analyse)")
    app.xfn_launch_kwargs = look if new_api else {}
    return app


def launch(app, **kwargs):
    """app.launch() with the theme passed where this Gradio version expects it."""
    extra = getattr(app, "xfn_launch_kwargs", {})
    try:
        return app.launch(**extra, **kwargs)
    except TypeError:          # a Gradio that does not take theme/js in launch()
        return app.launch(**kwargs)

## 3. Install Gradio

In [ ]:
!pip install -q gradio
import gradio as gr
print("gradio", gr.__version__)

## 4. Load the pipeline and pick examples

In [ ]:
import sys, pandas as pd
sys.path.insert(0, "src")
from xfakenet.pipeline import XFakeNetPipeline
from xfakenet.demo import build_app, launch

pipe = XFakeNetPipeline.load()
examples = []
test = pd.read_csv("data/processed/corpus_test.csv")
for label in ["Fabricated", "Misleading", "Credible"]:
    t = test[(test.label == label) & (test.text.str.split().str.len().between(10, 30))].sample(1, random_state=7)
    examples.append([t.text.iloc[0], None])
flags = "results/week3/modality_flags.csv"
if os.path.exists(flags):
    fl = pd.read_csv(flags)
    for f in ["CROSS-MODAL", "TEXT", "IMAGE"]:
        g = fl[(fl.modality_flag == f) & fl.image_path.map(os.path.exists)]
        if len(g):
            r = g.sample(1, random_state=7).iloc[0]
            examples.append([r.title, r.image_path])
out = pipe(examples[0][0])          # warm-up (loads the model)
print("Ready. Example verdict:", out["label"], "| highlighted:", out["highlighted"])

## 5. Launch the demo
Open the `https://….gradio.live` link printed below. Keep this tab open while you use or share it. To stop, press the ■ button on this cell.

In [ ]:
app = build_app(pipe, examples)
launch(app, share=True, debug=True, allowed_paths=[WORK])